In [ ]:
!pip install captum transformers datasets librosa

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from captum.attr import GradientShap

from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor
)

from datasets import load_dataset, Audio

import os
from scipy.interpolate import interp1d

### Δείγμα αναφοράς

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

model.eval()

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

###Πρόβλεψη μοντέλου για το δείγμα

In [ ]:
sample_index = TARGET_IDX

sample = test_dataset[sample_index]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

In [ ]:
inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")
input_values = inputs["input_values"]

with torch.no_grad():
    outputs = model(**inputs)

probabilities = outputs.logits
predicted_class = probabilities.argmax(-1).item()

print("True label:", true_label)
print("Predicted label:", model.config.id2label[predicted_class])

###Wrapper για forward pass

In [ ]:
#Wrapper για την επιστροφή raw logits του μοντέλου
class WrappedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()

        self.model = model

    def forward(self, input_values):
        outputs = self.model(input_values=input_values)

        return outputs.logits

wrapped_model = WrappedModel(model)
gs = GradientShap(wrapped_model)

###Υπολογισμός GradientSHAP

In [ ]:
# Μηδενικό baseline ίδιο με άλλες μεθόδους
baseline = torch.zeros_like(input_values)

attributions = gs.attribute(input_values, baselines=baseline, target=predicted_class)

In [ ]:
attr = (attributions.squeeze().detach().numpy())
audio_np = input_values.squeeze().numpy()

###Οπτικοποίηση Αποτελεσμάτων

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
axes[0].plot(audio_np, color='gray')
axes[0].set_title(f"Κυματομορφή — True label: {true_label}")

attr_abs = np.abs(attr)
attr_smooth = np.convolve(attr_abs, np.ones(400)/400, mode='same')
axes[1].plot(attr_smooth, color='red')
axes[1].set_title(f"GradientSHAP — Predicted: {model.config.id2label[predicted_class]}")

plt.tight_layout()
plt.show()

### Αποθήκευση attribution

In [ ]:
def resample_attribution(attr, target_frames=50):
    """
    Επαναδειγματοληπτεί την απόλυτη τιμή ενός attribution vector σε
    target_frames σημεία με γραμμική παρεμβολή, κανονικοποιώντας
    ώστε το άθροισμα να ισούται με 1. Χρησιμοποιείται επειδή το GradientSHAP
    παράγει attribution στο πλήρες μήκος του raw σήματος
    """
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

resampled_gradshap = resample_attribution(attr, target_frames=50)

np.save(f"{save_dir}/GradientSHAP_idx{TARGET_IDX}.npy", resampled_gradshap)
print(f"Αποθηκεύτηκε: GradientSHAP_idx{TARGET_IDX}.npy, shape={resampled_gradshap.shape}")

In [ ]:
resampled_gradshap = resample_attribution(attr, target_frames=50)

np.save(f"{save_dir}/GradientSHAP_idx{TARGET_IDX}.npy", resampled_gradshap)
print(f"Αποθηκεύτηκε: GradientSHAP_idx{TARGET_IDX}.npy, shape={resampled_gradshap.shape}")